# Lesson 4.8: Practice

**Analyst:** John Ko
**Date:** 11/11/26

## Imports and Setups

In [1]:
import pandas as pd
import numpy as np
import polars as pl
import duckdb
import pyarrow
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f"Polars: {pl.__version__}")
print(f"DuckDB: {duckdb.__version__}")
print(f"PyArrow: {pyarrow.__version__}")

Polars: 2.1.0
DuckDB: 2.0.0
PyArrow: 26.0.1


## Implementation 1: Pandas (reference)

In [2]:
# Function to calculate churn rate pct
def churn_rate_pct(active_series):
    return (~active_series).mean() * 100

# Record starting time
start = time.time()

# Load CSV using Pandas
df_pd = pd.read_csv(
    'finflow-enriched-200k.csv',
    parse_dates=['signup_date', 'last_login_date', 'churn_date']
)

# turn each date into its month
df_pd['signup_cohort'] = df_pd['signup_date'].dt.to_period('M')

pandas_result = df_pd.groupby('signup_cohort').agg(
    customer_count=('customer_id', 'count'),
    churn_rate_pct=('active', churn_rate_pct)
).round(2)

# Check how long to finish the code above.
pandas_time = time.time() - start
print(f"Pandas time: {pandas_time:.4f}s")
print(pandas_result)

Pandas time: 0.2842s
               customer_count  churn_rate_pct
signup_cohort                                
2024-01                  8303           20.41
2024-02                  7870           20.48
2024-03                  8542           20.44
2024-04                  8349           19.60
2024-05                  8601           20.44
2024-06                  8092           20.58
2024-07                  8494           20.19
2024-08                  8516           19.39
2024-09                  8069           19.31
2024-10                  8364           20.31
2024-11                  8301           20.32
2024-12                  8472           20.16
2025-01                  8458           19.26
2025-02                  7539           19.23
2025-03                  8424           19.75
2025-04                  8143           20.23
2025-05                  8476           19.71
2025-06                  8231           20.61
2025-07                  8459           20.09
2025-08      

## Implementation 2: Polars

In [3]:
start = time.time()

# Load CSV using polars with parse_date 
df_polars = pl.read_csv(
    'finflow-enriched-200k.csv',
    try_parse_dates=True
)


polars_result = (df_polars.with_columns(pl.col('signup_date').dt.truncate('1mo').alias('signup_cohort'))
    .group_by('signup_cohort')
    .agg(
        pl.col('customer_id').count().alias('customer_count'),
        (pl.col('active').not_().sum() / pl.col('active').count() * 100)
                 .round(2).alias('churn_rate_pct'),
    )
    .sort('signup_cohort')
)
                 
polars_time = time.time() - start
print(f'polars_time: {polars_time:.4f} s')
print(polars_result)

polars_time: 0.0269 s
shape: (25, 3)
┌───────────────┬────────────────┬────────────────┐
│ signup_cohort ┆ customer_count ┆ churn_rate_pct │
│ ---           ┆ ---            ┆ ---            │
│ date          ┆ u32            ┆ f64            │
╞═══════════════╪════════════════╪════════════════╡
│ 2024-01-01    ┆ 8303           ┆ 20.41          │
│ 2024-02-01    ┆ 7870           ┆ 20.48          │
│ 2024-03-01    ┆ 8542           ┆ 20.44          │
│ 2024-04-01    ┆ 8349           ┆ 19.6           │
│ 2024-05-01    ┆ 8601           ┆ 20.44          │
│ …             ┆ …              ┆ …              │
│ 2025-09-01    ┆ 8198           ┆ 19.9           │
│ 2025-10-01    ┆ 8724           ┆ 20.12          │
│ 2025-11-01    ┆ 8161           ┆ 19.51          │
│ 2025-12-01    ┆ 8493           ┆ 19.97          │
│ 2026-01-01    ┆ 287            ┆ 19.16          │
└───────────────┴────────────────┴────────────────┘


## Implementation 3: DuckDB SQL.

In [4]:
start = time.time()

duckdb_result = duckdb.sql("""
    SELECT
        DATE_TRUNC('month', signup_date) AS signup_cohort,
        COUNT(*) AS customer_count,
        ROUND(SUM(CASE WHEN active = false THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
    FROM read_csv_auto('finflow-enriched-200k.csv')
    GROUP BY DATE_TRUNC('month', signup_date)
    ORDER BY DATE_TRUNC('month', signup_date)
""").df()

duckdb_time = time.time() - start
print(f'duckdb_time: {duckdb_time:.4f} s')
print(duckdb_result)

duckdb_time: 0.1331 s
   signup_cohort  customer_count  churn_rate_pct
0     2024-01-01            8303           20.41
1     2024-02-01            7870           20.48
2     2024-03-01            8542           20.44
3     2024-04-01            8349           19.60
4     2024-05-01            8601           20.44
5     2024-06-01            8092           20.58
6     2024-07-01            8494           20.19
7     2024-08-01            8516           19.39
8     2024-09-01            8069           19.31
9     2024-10-01            8364           20.31
10    2024-11-01            8301           20.32
11    2024-12-01            8472           20.16
12    2025-01-01            8458           19.26
13    2025-02-01            7539           19.23
14    2025-03-01            8424           19.75
15    2025-04-01            8143           20.23
16    2025-05-01            8476           19.71
17    2025-06-01            8231           20.61
18    2025-07-01            8459           20.0

## Validation
Check if all tables are matching

In [5]:
print(pandas_result)
print(polars_result)
print(duckdb_result)

               customer_count  churn_rate_pct
signup_cohort                                
2024-01                  8303           20.41
2024-02                  7870           20.48
2024-03                  8542           20.44
2024-04                  8349           19.60
2024-05                  8601           20.44
2024-06                  8092           20.58
2024-07                  8494           20.19
2024-08                  8516           19.39
2024-09                  8069           19.31
2024-10                  8364           20.31
2024-11                  8301           20.32
2024-12                  8472           20.16
2025-01                  8458           19.26
2025-02                  7539           19.23
2025-03                  8424           19.75
2025-04                  8143           20.23
2025-05                  8476           19.71
2025-06                  8231           20.61
2025-07                  8459           20.09
2025-08                  8434     

In [6]:
print(f"Pandas:  {pandas_time:.4f}s")
print(f"Polars:  {polars_time:.4f}s")
print(f"DuckDB:  {duckdb_time:.4f}s")
print(f"\nFastest is {min((pandas_time, 'Pandas'), (polars_time, 'Polars'), (duckdb_time, 'DuckDB'))[1]}")

Pandas:  0.2842s
Polars:  0.0269s
DuckDB:  0.1331s

Fastest is Polars


The one that felt the most natural is DuckDB since I'm most familiar with the SQL syntax.
The one with the least cleanup would be DuckDB, with most of the results already handled within the query.
For Monday morning, with a 20-million-row table, I would use DuckDB; it is built for this with SQL and joining tables with a huge amount of data.